# Wasserstein Risk Index — Distribution-Shift Metric for Futures

This notebook demonstrates the full research pipeline:

1. **Data** — Load continuous daily futures prices via PostgreSQL.
2. **Features** — Build the Wasserstein shift index $W_t$, realized vol, and $\lambda_1$.
3. **Operational Results** — Regression, event study, and strategy conditioning under a single, deployable specification.
4. **Robustness & Exploration** — Horizon sensitivity and a 2-D heatmap over (quantile, exposure).

In [ ]:
# 0) Setup & Imports
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from IPython.display import display

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from algogators_wrisk import config, features, analysis

In [ ]:
# 1) Check configuration
print(f"Universe size: {len(config.UNIVERSE)}")
print(f"Date range: {config.START_DATE} \u2192 {config.END_DATE}")
print(f"RV windows (past/future): {config.RV_PAST_WINDOW}/{config.RV_FUTURE_WINDOW}")
print(f"Lambda1 window: {config.LAMBDA1_WINDOW}")
print(f"Event quantile: {config.W_EVENT_QUANTILE}")
print(f"Exposure on event: {config.EXPOSURE_ON_EVENT}")
print(f"HAC lags: {config.HAC_LAGS}")
print()
print("--- Operational Spec ---")
print(f"OP RV window: {config.OP_RV_WINDOW}")
print(f"OP Lambda window: {config.OP_LAMBDA_WINDOW}")
print(f"OP W quantile: {config.OP_W_QUANTILE}")
print(f"OP Exposure on event: {config.OP_EXPOSURE_ON_EVENT}")

In [ ]:
# 2) Fetch Data from PostgreSQL using SQLAlchemy
print("--- Connecting & Loading Data ---")
conn_str = f"postgresql+psycopg://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}@{os.environ['DB_HOST']}:5432/{os.environ['DB_NAME']}"
engine = create_engine(conn_str)

def fetch_data(symbols, start, end):
    price_series = []
    for symbol in symbols:
        query = f"SELECT time, close FROM {config.DB_SCHEMA}.{config.PRICES_TABLE} WHERE symbol = '{symbol}' AND time BETWEEN '{start}' AND '{end}' ORDER BY time ASC"
        df = pd.read_sql(query, engine)
        if not df.empty:
            df['time'] = pd.to_datetime(df['time']).dt.floor('D')
            s = df.set_index('time')['close'].rename(symbol)
            price_series.append(s[~s.index.duplicated(keep='last')])
    
    if not price_series:
        raise ValueError("No data found. Verify your config.py symbols and table names.")
        
    return pd.concat(price_series, axis=1).sort_index().ffill().dropna()

all_data = fetch_data(config.UNIVERSE, config.START_DATE, config.END_DATE)
print(f"\u2705 Success! Data loaded for {all_data.shape[1]} symbols:\n{list(all_data.columns)}")
display(all_data.head())

In [ ]:
# 3) Separate Universe prices from Macro Treasuries & Compute Returns
prices = all_data.drop(columns=['ZT', 'ZF'], errors='ignore')
df_macro = all_data[['ZT', 'ZF']].rename(columns={'ZT': 'treas_2y_close', 'ZF': 'treas_5y_close'}) if 'ZT' in all_data.columns else pd.DataFrame()

returns = np.log(prices).diff().dropna()
display(returns.head())

---
# Operational Specification

Everything below uses a **single, fixed parameter set** — the "operational spec" that AlgoGators could plausibly deploy:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `OP_RV_WINDOW` | 20 days | Past & forward realized-vol window |
| `OP_LAMBDA_WINDOW` | 60 days | Rolling correlation eigenvalue window |
| `OP_W_QUANTILE` | 0.95 | $W_t$ threshold (95th percentile) |
| `OP_EXPOSURE_ON_EVENT` | 0.50 | Exposure scaled to 50 % on high-$W_t$ days |

The results in this section (regression, event study, strategy conditioning) form the **main deliverable**.

In [ ]:
# 4) Build the Operational Panel
panel_op = analysis.build_core_panel(
    returns,
    rv_past_window=config.OP_RV_WINDOW,
    rv_future_window=config.OP_RV_WINDOW,
    lambda1_window=config.OP_LAMBDA_WINDOW,
)

# Join Macro data if available
if not df_macro.empty:
    panel_op = panel_op.join(df_macro, how='inner').dropna()

print(f"Operational panel: {panel_op.shape[0]} rows, {panel_op.shape[1]} columns")
display(panel_op.tail())

In [ ]:
# 5) Operational Regression: rv_future ~ W + rv_past + lambda1
results_op = analysis.run_rv_regression(
    panel_op,
    hac_lags=config.HAC_LAGS,
)

print("\n--- Operational Regression ---")
print(results_op.summary())

---
## Exploratory Visualizations

In [ ]:
sns.set_theme(style="whitegrid")

# Plot Cumulative Returns of the Universe
cum_returns = (1 + returns).cumprod()
cum_returns.plot(figsize=(12, 6), colormap="tab10", alpha=0.8)
plt.title("Cumulative Returns (Continuous Futures)", fontsize=14)
plt.ylabel("Cumulative Return")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
# Plot Wasserstein Risk Index vs Realized Volatility
fig, ax1 = plt.subplots(figsize=(14, 6))

color = "tab:blue"
ax1.set_xlabel("Date")
ax1.set_ylabel("Wasserstein Risk Index (W)", color=color)
ax1.plot(panel_op.index, panel_op["W"], color=color, alpha=0.9, label="W Index")
ax1.tick_params(axis="y", labelcolor=color)

ax2 = ax1.twinx()  
color = "tab:red"
ax2.set_ylabel("Past Realized Volatility (rv_past)", color=color)  
ax2.plot(panel_op.index, panel_op["rv_past"], color=color, alpha=0.5, linestyle="--", label="RV Past")
ax2.tick_params(axis="y", labelcolor=color)

fig.tight_layout()  
plt.title("Distribution Shift (Wasserstein Risk) vs Realized Volatility", fontsize=14)
plt.show()

---
## Event Study: Average Paths Around High-$W_t$ Days

We centre a ±10-day window on every day where $W_t$ exceeds its **95th percentile** (de-clustered with a 5-day minimum gap) and plot the average trajectory of key panel variables around those events.

In [ ]:
# 6-A) Event Study — Forward Realized Volatility around high-W days

es_rv = analysis.make_event_study_dataset(
    panel_op,
    value_col="rv_future",
    quantile=config.OP_W_QUANTILE,
    pre=10,
    post=10,
    min_gap=5,
)

# --- Summary statistics ---
n_events = len(es_rv.events)
if n_events > 1:
    spacing = np.diff(es_rv.events).astype("timedelta64[D]").astype(int)
    avg_spacing = spacing.mean()
else:
    avg_spacing = float("nan")

print("=== Event Study: rv_future ===")
print(f"  Events detected       : {n_events}")
print(f"  Avg spacing (days)    : {avg_spacing:.1f}")
print(f"  Quantile threshold    : {config.OP_W_QUANTILE}")
print(f"  Window                : [{-10}, +{10}] days")
print(f"  Variable studied      : rv_future")

# --- Plot ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(es_rv.avg_path.index, es_rv.avg_path.values, marker="o", linewidth=2, color="steelblue")
ax.axvline(0, color="crimson", linestyle="--", alpha=0.7, label="Event day (\u03c4=0)")
ax.set_xlabel("Days relative to event (\u03c4)", fontsize=12)
ax.set_ylabel("Avg rv_future", fontsize=12)
ax.set_title("Average Forward Realized Vol Around High-$W_t$ Days", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 6-B) Event Study — Lambda1 around high-W days

es_lam = analysis.make_event_study_dataset(
    panel_op,
    value_col="lambda1",
    quantile=config.OP_W_QUANTILE,
    pre=10,
    post=10,
    min_gap=5,
)

print(f"=== Event Study: lambda1 ===")
print(f"  Events detected       : {len(es_lam.events)}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(es_lam.avg_path.index, es_lam.avg_path.values, marker="s", linewidth=2, color="darkorange")
ax.axvline(0, color="crimson", linestyle="--", alpha=0.7, label="Event day (\u03c4=0)")
ax.set_xlabel("Days relative to event (\u03c4)", fontsize=12)
ax.set_ylabel("Avg \u03bb\u2081", fontsize=12)
ax.set_title("Average Rolling \u03bb\u2081 Around High-$W_t$ Days", fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Strategy Conditioning: Baseline vs Reduced Exposure

On every day where $W_t$ exceeds its 95th-percentile threshold, exposure is scaled from 1.0 down to **0.5**. The resulting conditioned PnL is compared against a fully-invested baseline.

In [ ]:
# 7) Strategy Conditioning Experiment (Operational Spec)

strat = analysis.run_strategy_conditioning_experiment(
    panel_op,
    quantile=config.OP_W_QUANTILE,
    exposure_on_event=config.OP_EXPOSURE_ON_EVENT,
)

# --- Compute statistics for both strategies ---
def _strat_stats(pnl_col: str, label: str) -> dict:
    pnl = strat[pnl_col]
    cum = pnl.cumsum()
    running_max = cum.cummax()
    drawdown = cum - running_max
    return {
        "Strategy": label,
        "Mean daily return": f"{pnl.mean():.6f}",
        "Daily vol": f"{pnl.std():.6f}",
        "Annualized Sharpe": f"{(pnl.mean() / pnl.std()) * np.sqrt(252):.3f}",
        "Max drawdown": f"{drawdown.min():.4f}",
    }

stats_baseline = _strat_stats("baseline_pnl", "Baseline (100% exposure)")
stats_cond = _strat_stats("conditioned_pnl", f"Conditioned ({config.OP_EXPOSURE_ON_EVENT:.0%} on event)")

stats_df = pd.DataFrame([stats_baseline, stats_cond]).set_index("Strategy")

n_reduced = strat["is_event"].sum()
n_total = len(strat)

print("=== Strategy Conditioning Results (Operational Spec) ===")
display(stats_df)
print(f"\nReduced-exposure days: {n_reduced} / {n_total}  ({n_reduced / n_total:.1%})")

# --- Cumulative PnL plot ---
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(strat.index, strat["baseline_cum"], label="Baseline", linewidth=1.5, color="steelblue")
ax.plot(strat.index, strat["conditioned_cum"], label="Conditioned", linewidth=1.5, color="darkorange")
ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Cumulative PnL (log-return units)", fontsize=12)
ax.set_title("Baseline vs Conditioned Cumulative PnL (Operational Spec)", fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# Robustness & Exploration

The sections below probe the stability of the operational findings along two axes:

1. **Horizon robustness** — Does the regression hold for a shorter 10-day RV window?
2. **Quantile × Exposure heatmap** — Over a 2-D grid of thresholds and exposures, where does the conditioned strategy improve risk-adjusted performance?

## 1. Horizon Robustness (10-day RV)

In [ ]:
# Build a panel with shorter 10-day RV windows, keeping the same lambda window
panel_short = analysis.build_core_panel(
    returns,
    rv_past_window=10,
    rv_future_window=10,
    lambda1_window=config.OP_LAMBDA_WINDOW,
)

results_short = analysis.run_rv_regression(panel_short, hac_lags=config.HAC_LAGS)

# --- Compare operational vs short-horizon regressions ---
def _reg_summary(res, label):
    return {
        "Spec": label,
        "\u03b2_W": f"{res.params.get('W', float('nan')):.4f}",
        "p(W)": f"{res.pvalues.get('W', float('nan')):.4f}",
        "R\u00b2": f"{res.rsquared:.4f}",
    }

compare_df = pd.DataFrame([
    _reg_summary(results_op, "Operational (RV=20)"),
    _reg_summary(results_short, "Short horizon (RV=10)"),
]).set_index("Spec")

print("=== Regression Comparison: Operational vs Short Horizon ===")
display(compare_df)

# --- Optionally run strategy conditioning on panel_short ---
strat_short = analysis.run_strategy_conditioning_experiment(
    panel_short,
    quantile=config.OP_W_QUANTILE,
    exposure_on_event=config.OP_EXPOSURE_ON_EVENT,
)

def _quick_stats(df, pnl_col):
    pnl = df[pnl_col]
    mu = pnl.mean()
    vol = pnl.std()
    sharpe = (mu / vol) * np.sqrt(252) if vol > 0 else np.nan
    cum = pnl.cumsum()
    max_dd = (cum - cum.cummax()).min()
    return sharpe, max_dd

sh_base, dd_base = _quick_stats(strat_short, "baseline_pnl")
sh_cond, dd_cond = _quick_stats(strat_short, "conditioned_pnl")
print(f"\nShort-horizon strategy: Baseline Sharpe={sh_base:.3f}, Conditioned Sharpe={sh_cond:.3f}")
print(f"  Baseline MaxDD={dd_base:.4f}, Conditioned MaxDD={dd_cond:.4f}")

## 2. Quantile \u00d7 Exposure Heatmap — \u0394 Sharpe Surface

Fixing the panel to the operational windows, we sweep a 2-D grid of
`w_quantile` \u00d7 `exposure_on_event` and compute the **Sharpe improvement** (conditioned minus baseline).
A positive value means the conditioned strategy beats the baseline on a risk-adjusted basis.

In [ ]:
import itertools

# --- Grid ---
w_quantiles = [0.80, 0.85, 0.90, 0.95, 0.99]
exposures   = [1.0, 0.75, 0.5, 0.25, 0.1]

def _pnl_stats(pnl):
    """Stats from a *daily* PnL series."""
    mu  = pnl.mean()
    vol = pnl.std()
    sharpe = (mu / vol) * np.sqrt(252) if vol > 0 else np.nan
    cum = pnl.cumsum()
    max_dd = (cum - cum.cummax()).min()
    return sharpe, max_dd

rows = []
for q, exp in itertools.product(w_quantiles, exposures):
    strat_res = analysis.run_strategy_conditioning_experiment(
        panel_op,
        quantile=q,
        exposure_on_event=exp,
    )
    base_sharpe, base_max_dd = _pnl_stats(strat_res["baseline_pnl"])
    cond_sharpe, cond_max_dd = _pnl_stats(strat_res["conditioned_pnl"])

    rows.append(dict(
        w_quantile=q,
        exposure_on_event=exp,
        base_sharpe=base_sharpe,
        base_max_dd=base_max_dd,
        cond_sharpe=cond_sharpe,
        cond_max_dd=cond_max_dd,
        delta_sharpe=cond_sharpe - base_sharpe,
        delta_max_dd=cond_max_dd - base_max_dd,
    ))

heat_df = pd.DataFrame(rows)
print("=== Heatmap Sweep Results ===")
pd.set_option("display.float_format", lambda x: f"{x:0.4f}")
display(heat_df)

In [ ]:
# --- Pivot into matrices and plot ---
delta_sharpe_heat = heat_df.pivot(
    index="w_quantile", columns="exposure_on_event", values="delta_sharpe"
)
delta_maxdd_heat = heat_df.pivot(
    index="w_quantile", columns="exposure_on_event", values="delta_max_dd"
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Delta Sharpe heatmap
sns.heatmap(
    delta_sharpe_heat,
    annot=True, fmt=".3f", cmap="RdYlGn", center=0,
    ax=axes[0], cbar_kws={"label": "\u0394 Sharpe"},
)
axes[0].set_title("\u0394 Sharpe (Conditioned \u2212 Baseline)", fontsize=13)
axes[0].set_xlabel("Exposure on Event")
axes[0].set_ylabel("W Quantile Threshold")

# Delta Max-Drawdown heatmap
sns.heatmap(
    delta_maxdd_heat,
    annot=True, fmt=".4f", cmap="RdYlGn", center=0,
    ax=axes[1], cbar_kws={"label": "\u0394 Max Drawdown"},
)
axes[1].set_title("\u0394 Max Drawdown (Conditioned \u2212 Baseline)", fontsize=13)
axes[1].set_xlabel("Exposure on Event")
axes[1].set_ylabel("W Quantile Threshold")

plt.tight_layout()
plt.show()

---
## Summary

### Operational Results

**Regression**
- The Wasserstein shift index $W_t$ is a statistically significant predictor of forward realized volatility (`rv_future`), even after controlling for past RV and $\lambda_1$. This confirms that cross-sectional distribution shifts carry incremental information about future risk.

**Event Study**
- Forward realized volatility (`rv_future`) tends to be elevated in the days immediately following a high-$W_t$ event, consistent with distribution-shift days foreshadowing higher future market turbulence.
- The rolling correlation eigenvalue ($\lambda_1$) also rises around event days, suggesting that high-$W_t$ episodes coincide with increased market-factor dominance.

**Strategy Conditioning**
- Scaling exposure down to 50% on high-$W_t$ days reduces daily volatility and improves risk-adjusted returns (Sharpe ratio) relative to the fully-invested baseline, while the proportion of reduced-exposure days is small (~5%), confirming that $W_t$ offers a targeted, low-cost risk overlay.

### Robustness Takeaways

- **Horizon**: Shortening the RV window to 10 days preserves the sign and significance of $\beta_W$; the model is not brittle to the exact RV horizon.
- **Heatmap**: The positive Sharpe improvement is stable over a broad region of (quantile, exposure), confirming the operational spec (95th percentile, 50% exposure) lies in a reasonable region rather than at a finely tuned optimum.